In [18]:
from vllm import LLM, SamplingParams
from utils import load_jsonl
from typing import Iterable, List,Callable
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn, question_only_reward_fn
from utils import *
from vllm_utils import *
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

# Load LLM

In [3]:
model_path = '../model/GRPO'
llm = init_vllm(model_path,device=get_device(4),seed=42,gpu_memory_utilization=0.3)
print('init LLM successfully!')

INFO 03-14 13:04:32 config.py:542] This model supports multiple tasks: {'classify', 'reward', 'generate', 'embed', 'score'}. Defaulting to 'generate'.
INFO 03-14 13:04:32 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/GRPO', speculative_config=None, tokenizer='../model/GRPO', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda:4, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/GRPO, num_scheduler_steps=1, multi_step_stream_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-14 13:04:37 model_runner.py:1115] Loading model weights took 0.0000 GB
INFO 03-14 13:04:39 worker.py:267] Memory profiling takes 1.84 seconds
INFO 03-14 13:04:39 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.30) = 7.08GiB
INFO 03-14 13:04:39 worker.py:267] model weights take 0.00GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 0.00GiB; the rest of the memory reserved for KV Cache is 7.08GiB.
INFO 03-14 13:04:40 executor_base.py:110] # CUDA blocks: 16560, # CPU blocks: 9362
INFO 03-14 13:04:40 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 64.69x
INFO 03-14 13:04:46 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:47<00:00,  1.37s/it]

INFO 03-14 13:05:34 model_runner.py:1562] Graph capturing finished in 48 secs, took 0.00 GiB
INFO 03-14 13:05:34 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 56.77 seconds


init LLM successfully!


In [23]:
# load_template
template_path = '../cs336_alignment/prompts/r1_zero.prompt'
r1_template = load_template(template_path)
questions = ["Among all pairs of real numbers $(x, y)$ such that $\sin \sin x = \sin \sin y$ with $-10\pi \le x, y \le 10\pi$, Oleg randomly selected a pair $(X, Y)$. Compute the probability that $X = Y$.",
"A coin is tossed repeatedly until it lands heads for the first time. Given that the coin is tossed at least twice, what is the probability that it is tossed exactly three times?",
"Triangle $ABC$ has side lengths $AB = 5$, $BC = 6$, and $CA = 7$. Two different points $P$ and $Q$ lie on $\overline{BC}$, such that $\angle BAP = \angle PAQ = \angle QAC$. Compute $\frac{PQ}{BC}$.",
"How many positive integers less than $1000$ are divisible by $3$ or $5$ but not by $15$?",
"A bag contains $4$ red marbles, $5$ white marbles, and $6$ blue marbles. Three marbles are drawn at random without replacement. What is the probability that they are all the same color?"]
questions_format = [r1_template.format(question=q) for q in questions]
# r1_template
questions[1]

'A coin is tossed repeatedly until it lands heads for the first time. Given that the coin is tossed at least twice, what is the probability that it is tossed exactly three times?'

In [24]:
# generation
import os 
# evaluation
sampling_params = SamplingParams(
temperature=1.0,
top_p=1.0,
max_tokens=1024,
min_tokens=4,
stop = ["</answer>"],
include_stop_str_in_output = True
)
responses = generate_responses(llm, prompts = questions_format,sampling_params=sampling_params)

Processed prompts: 100%|██████████| 5/5 [00:03<00:00,  1.28it/s, est. speed input: 184.78 toks/s, output: 415.18 toks/s]


In [25]:
questions[1],responses[1]

('A coin is tossed repeatedly until it lands heads for the first time. Given that the coin is tossed at least twice, what is the probability that it is tossed exactly three times?',
 " Let's denote the event that the coin is tossed exactly \\(n\\) times as \\(A_n\\). The probability that the coin is tossed exactly \\(n\\) times is the probability that the first \\(n-1\\) tosses are tails and the \\(n\\)-th toss is heads. This is \\(\\left(\\frac{1}{2}\\right)^n\\). We need to find the probability that the coin is tossed exactly three times given that it is tossed at least twice. This is \\(P(A_3 | \\text{at least 2 tosses}) = \\frac{P(A_3)}{P(\\text{at least 2 tosses})}\\). The probability \\(P(A_3) = \\left(\\frac{1}{2}\\right)^3 = \\frac{1}{8}\\). The probability that the coin is tossed at least twice is \\(P(\\text{at least 2 tosses}) = 1 - P(\\text{tossed less than 2 times}) = 1 - \\left(1 + \\frac{1}{2}\\right) = \\frac{1}{2}\\). So, \\(P(A_3 | \\text{at least 2 tosses}) = \\frac{

### 问题：  
反复抛一枚硬币，直到首次出现正面为止。已知这枚硬币至少被抛了两次，求它恰好被抛了三次的概率。
### 答案：  
设事件 $A_n$ 表示硬币恰好被抛了 $n$ 次。硬币恰好被抛 $n$ 次的概率是前 $n-1$ 次都是反面且第 $n$ 次是正面的概率，即 $\left(\frac{1}{2}\right)^n$。我们需要求在已知至少抛两次的条件下，恰好抛三次的概率，即  
$
P(A_3 \mid \text{至少抛两次}) = \frac{P(A_3)}{P(\text{至少抛两次})}.
$  其中 $P(A_3) = \left(\frac{1}{2}\right)^3 = \frac{1}{8}$。  
至少抛两次的概率为  $
P(\text{至少抛两次}) = 1 - P(\text{抛少于两次}) = 1 - \left(P(A_1) + P(A_0)\right),
$  但注意这里没有抛 0 次的情况，所以是  $
P(\text{至少抛两次}) = 1 - P(\text{第一次抛就出现正面}) = 1 - \frac{1}{2} = \frac{1}{2}.$  
因此  $
P(A_3 \mid \text{至少抛两次}) = \frac{\frac{1}{8}}{\frac{1}{2}} = \frac{1}{4}.
$  答案是 $\boxed{\frac{1}{4}}$。